# 07 - BAF Cross-Dataset Rule and Explanation Replication

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ.setdefault("THESIS_QUICK_RUN", "0")
    os.environ.setdefault("THESIS_SYNTHETIC_FALLBACK", "0")

def find_project_root() -> Path | None:
    direct_candidates = [KAGGLE_PROJECT_DIR, Path.cwd(), *Path.cwd().parents]
    for candidate in direct_candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = sorted(path.parent for path in base.glob("**/configs") if path.is_dir())
            for candidate in matches:
                if (candidate / "src").is_dir():
                    return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and KAGGLE:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
        check=True,
    )
    PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

AUDIT_SOURCE_FILES = (
    "scripts/generate_notebooks.py",
    "src/artifacts.py",
    "src/data/dataset.py",
    "src/data/preprocessing.py",
    "src/experiment.py",
    "src/explanation/explanation_metrics.py",
    "src/explanation/rule_explainer.py",
    "src/logic/fraud_rules.py",
    "src/logic/knowledge_base.py",
    "src/logic/predicates.py",
    "src/logic/tensor_logic.py",
)

def audit_pipeline_fingerprint(config_path: Path) -> str:
    paths = [PROJECT_ROOT / relative for relative in AUDIT_SOURCE_FILES]
    paths.append(Path(config_path))
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Files required for the audit-pipeline fingerprint are missing: {missing}")
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.relative_to(PROJECT_ROOT).as_posix()):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Notebook chỉ đọc frozen BAF reference predictor từ Notebook 03. Rules fit trên train months 0-4,
được kiểm tra trên validation month 5 và locked test months 6-7.
Đây là replication/portability evaluation với BAF-specific predictor và rule base, không phải
việc chuyển nguyên model hoặc rules từ IEEE-CIS. BAF là privacy-preserving synthetic benchmark.

In [ ]:
preflight_manifests = []
preflight_artifacts = []
for root_value in INPUT_ROOTS:
    root = Path(root_value)
    if not root.exists():
        continue
    manifest_paths = [root] if root.is_file() and root.name == "frozen_reference_manifest.json" else list(root.glob("**/frozen_reference_manifest.json"))
    for manifest_path in manifest_paths:
        try:
            manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if str(manifest_payload.get("dataset_name", "")).lower() != "baf".lower():
            continue
        preflight_manifests.append(manifest_path.resolve())
        artifact_path = manifest_path.parent / str(manifest_payload.get("artifact_file", ""))
        if artifact_path.exists():
            preflight_artifacts.append(artifact_path.resolve())

preflight_table = pd.DataFrame({
    "manifest": [str(path) for path in sorted(set(preflight_manifests))],
})
display(preflight_table)
print("Frozen artifacts:")
for path in sorted(set(preflight_artifacts)):
    print(path)
if not preflight_manifests or not preflight_artifacts:
    raise FileNotFoundError(
        "No complete baf frozen artifact was found below INPUT_ROOTS. "
        "On Kaggle, attach the corresponding benchmark notebook output; locally, place it below results/runs/notebooks."
    )

In [ ]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact, sha256_file
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/baf.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "baf", expected_config=config,
    search_roots=[OUTPUT_BASE / "03_baf_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
if int(artifact["manifest"]["reference_seed"]) != int(config["evaluation"]["reference_seed"]):
    raise ValueError("Frozen artifact reference seed does not match the locked dataset protocol")
if not QUICK_RUN and str(artifact["manifest"].get("data_source", "")).lower() == "synthetic":
    raise ValueError("Full thesis evaluation cannot consume a synthetic-fallback frozen artifact")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({
    "data_source": data_source,
    "benchmark_type": frame.attrs.get("benchmark_type", "privacy_preserving_synthetic_benchmark"),
    "reference_model": artifact["manifest"]["model"],
    "reference_seed": artifact["manifest"]["reference_seed"],
    "calibration_method": artifact["manifest"]["calibration_method"],
    "threshold": threshold,
    "artifact": str(artifact["artifact_path"]),
})

def write_upstream_lineage(destination, notebook_id, output_files, config_path):
    output_files = list(output_files)
    lineage = {
        "notebook_id": notebook_id,
        "git_commit": GIT_COMMIT,
        "dataset_name": artifact["manifest"]["dataset_name"],
        "data_source": data_source,
        "frozen_data_source": artifact["manifest"].get("data_source"),
        "quick_run": QUICK_RUN,
        "reference_model_key": artifact["manifest"]["model_key"],
        "reference_seed": artifact["manifest"]["reference_seed"],
        "config_sha256": artifact["manifest"]["config_sha256"],
        "audit_source_sha256": audit_pipeline_fingerprint(config_path),
        "frozen_manifest_sha256": sha256_file(artifact["manifest_path"]),
        "frozen_artifact_sha256": sha256_file(artifact["artifact_path"]),
        "output_files": output_files,
        "output_sha256": {
            name: sha256_file(Path(destination) / name) for name in output_files
        },
    }
    lineage_path = Path(destination) / "upstream_lineage.json"
    lineage_path.write_text(json.dumps(lineage, indent=2), encoding="utf-8")
    return lineage_path

## Rule and explanation results

In [ ]:
from src.explanation import (
    RuleExplainer, bootstrap_explanation_precision_gain,
    explanation_quality_metrics, rule_quality_table,
)
from src.logic import FraudKnowledgeBase, FraudRuleEngine

output_dir = OUTPUT_BASE / "07_baf_ltn_generalization"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
knowledge_base = FraudKnowledgeBase(engine)
fitted_thresholds = engine.fitted_thresholds()
rationale_rows = []
for definition in config["logic"]["rules"]:
    operators = [condition["operator"] for condition in definition["conditions"]]
    rationale_rows.append({
        "rule": definition["name"],
        "description": definition.get("description", ""),
        "knowledge_type": (
            "data-informed fuzzy hypothesis" if "category_risk" in operators
            else "domain hypothesis with train-fitted thresholds"
            if any(operator.endswith("_quantile") for operator in operators)
            else "configured domain hypothesis"
        ),
        "threshold_source": (
            "training-split quantile plus any explicit configured condition"
            if any(operator.endswith("_quantile") for operator in operators)
            else "explicit configured value"
        ),
        "limitation": (
            "BAF-specific association; this rule is not transferred from IEEE-CIS and is not causal."
        ),
    })
rule_rationale = pd.DataFrame(rationale_rows)
validation_truth = engine.evaluate(prepared.validation_frame)
test_truth = engine.evaluate(prepared.test_frame)
activation = float(config["logic"]["activation_threshold"])
validation_quality = rule_quality_table(validation_truth, prepared.y_validation, activation).assign(split="validation")
test_quality = rule_quality_table(test_truth, prepared.y_test, activation).assign(split="test")
rule_quality = pd.concat([validation_quality, test_quality], ignore_index=True)
satisfaction = pd.DataFrame([
    {"split": "train", **knowledge_base.satisfaction_breakdown(prepared.train_frame, target)},
    {"split": "validation", **knowledge_base.satisfaction_breakdown(prepared.validation_frame, target)},
    {"split": "test", **knowledge_base.satisfaction_breakdown(prepared.test_frame, target)},
])
explainer = RuleExplainer(engine, activation, config["logic"]["top_k_rules"])
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations, prepared.y_test, probabilities, threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"], seed=config["project"]["seed"],
))
explanation_quality = pd.DataFrame([quality])
display(
    fitted_thresholds,
    rule_rationale,
    rule_quality.round(4),
    satisfaction.round(4),
    explanation_quality.round(4),
)
rule_quality.to_csv(output_dir / "baf_rule_quality.csv", index=False)
satisfaction.to_csv(output_dir / "baf_knowledge_base_satisfaction.csv", index=False)
explanation_quality.to_csv(output_dir / "baf_explanation_quality.csv", index=False)
fitted_thresholds.to_csv(output_dir / "baf_fitted_rule_thresholds.csv", index=False)
rule_rationale.to_csv(output_dir / "baf_rule_rationale.csv", index=False)

In [ ]:
stability = validation_quality.merge(test_quality, on="rule", suffixes=("_validation", "_test"))
stability["coverage_delta"] = stability["coverage_test"] - stability["coverage_validation"]
stability["lift_delta"] = stability["lift_test"] - stability["lift_validation"]
display(stability[["rule", "coverage_delta", "lift_delta"]].round(4))
stability.to_csv(output_dir / "baf_rule_stability.csv", index=False)
lineage_path = write_upstream_lineage(
    output_dir,
    "07_BAF_Cross_Dataset_Rule_and_Explanation_Replication",
    [
        "baf_rule_quality.csv", "baf_knowledge_base_satisfaction.csv",
        "baf_explanation_quality.csv", "baf_fitted_rule_thresholds.csv",
        "baf_rule_rationale.csv", "baf_rule_stability.csv",
    ],
    PROJECT_ROOT / "configs/baf.yaml",
)
print({"upstream_lineage": str(lineage_path)})

## Takeaways

In [ ]:
eligible_rules = test_quality.dropna(subset=["lift"]).query("active_count > 0")
strongest_rule = eligible_rules.sort_values("lift", ascending=False).iloc[0]
display(Markdown(
    f"- Frozen predictor: **{artifact['manifest']['model']}**, seed **{artifact['manifest']['reference_seed']}**.\n"
    f"- Highest observed BAF test lift: **{strongest_rule['rule']} = {strongest_rule['lift']:.3f}** "
    f"with **{int(strongest_rule['active_count'])}** active rows.\n"
    f"- Alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**; "
    f"unsupported-alert rate: **{quality['unsupported_alert_rate']:.3f}**.\n"
    "- This is BAF-specific framework replication. It does not establish model/rule transfer, causality, or production generalization."
))